# Hybrid CNN-LSTM Stock Price Predictor Model

In [34]:
import pandas as pd
import numpy as np
from keras.layers import Input, Conv1D, Dense, BatchNormalization, Dropout, LSTM
from keras.models import Model

## Data

In [35]:
df = pd.read_csv("data/normalized_data.csv")
df.drop(columns=["date"], inplace=True)

In [47]:
X, y = [], []
features_list = [
    "trend_dpo",
    "oil_change_percent",
    "volume_em",
    "RTX_CMF",
    "japan_volume",
    "oil_volume",
    "lmt_rsi",
    "pltr_volume",
    "lmt_MACD_signal",
    "uk_volume",
    "lmt_volume",
    "lmt_MACD",
    "rtx_volume",
    "momentum_pvo",
    "RTX_RSI",
    "noc_volume",
    "QQQ_rsi",
    "volatility_dcp",
    "oil_rsi",
    "qqq_volume"
]
features_df = df[features_list].values
target_nums = df["target"].values
window_size = 20
num_features = len(features_list)

# Convert data into 3D for CNN (samples, timesteps, features)
for i in range(0, len(df) - window_size): 
        X.append(features_df[i : i + window_size]) # Add a 2D section of data of all features from the days within the current window size
        y.append(target_nums[i + window_size]) # Add the labels for each day in the window size

X = np.array(X)
y = np.array(y)

X.shape # Shape should be ((len(df) - window_size), window_size, num_features)
y.shape # Shape should be ((len(df) - window_size),)


(1108,)

In [43]:
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

## Model

All numbers are not real yet

In [48]:
inputs = Input(shape=(window_size,num_features))

# CNN Block
conv1 = Conv1D(filters=64, kernel_size=3, activation='relu')(inputs)
conv1 = BatchNormalization()(conv1)
conv1 = Dropout(0.4)(conv1)

conv2 = Conv1D(filters=64, kernel_size=5, activation='relu')(conv1)
conv2 = BatchNormalization()(conv2)
conv2 = Dropout(0.4)(conv2)

# LSTM Block
lstm = LSTM(5)(conv2)

# Dense layers and output
dense1 = Dense(5, activation='relu')(lstm)
outputs = Dense(3, activation='softmax')(dense1)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 20, 20)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_12 (Conv1D)              │ (None, 18, 64)         │         3,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 18, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_13 (Conv1D)              │ (None, 14, 64)         │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 14, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 5)              │         1,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 5)              │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,408 (103.16 KB)

 Trainable params: 26,152 (102.16 KB)

 Non-trainable params: 256 (1.00 KB)

In [50]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(X_train, y_train, batch_size=128, epochs=20)

score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6964 - loss: 0.5638
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7144 - loss: 0.5441 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7099 - loss: 0.5441 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7133 - loss: 0.5422 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7201 - loss: 0.5427 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7257 - loss: 0.5336 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7111 - loss: 0.5391 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7314 - loss: 0.5396 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7065 - loss: 0.5419 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7190 - loss: 0.5333 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7223 - loss: 0.5331 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7302 - loss: 0.5265 
E

In [ ]:
import numpy as np

sequence = np.random.random((100,))
print(sequence)

a = [1, 2, 3, 5, 6, 6, 7]
a[::2]

[0.04020226 0.26177759 0.02526432 0.647032   0.50276227 0.86155266
 0.01665974 0.14126553 0.10352367 0.39891908 0.47720508 0.49684888
 0.92320689 0.22224854 0.43543398 0.02743685 0.11467707 0.02618829
 0.57549919 0.37332888 0.81458192 0.95930012 0.33030297 0.81480943
 0.70974183 0.52728543 0.73618186 0.74102854 0.260343   0.91586149
 0.86063671 0.57577626 0.91352791 0.48032784 0.23928616 0.01286292
 0.26974456 0.80596291 0.18964498 0.74749446 0.1706248  0.08295924
 0.80636717 0.01006228 0.31469753 0.0090973  0.40067859 0.8789725
 0.7714201  0.70732206 0.29284832 0.71323466 0.13160575 0.08145188
 0.1804711  0.27385292 0.26284953 0.68044277 0.88791643 0.84773304
 0.89415999 0.78135864 0.23875774 0.62251441 0.2894332  0.82703156
 0.74263054 0.22225126 0.15363156 0.80877564 0.53263993 0.3473915
 0.60149476 0.67314395 0.86307149 0.17158113 0.45278236 0.78115381
 0.55050819 0.69129108 0.70831414 0.40297697 0.24732796 0.65975097
 0.71164362 0.28014954 0.81484466 0.68186021 0.45523507 0.664741

[1, 3, 6, 7]

## Training Model